# ISOT DistilBERT — Combined Corpus dış testi

Bu notebook yeni eğitim yapmaz. Reuters ifadesi çıkarılmış ISOT verisiyle eğittiğiniz modeli seçip Combined Corpus test dosyaları üzerinde değerlendirir. GPU seçin ve hücreleri sırayla çalıştırın. Etiket eşlemesi: 0 = gerçek, 1 = sahte.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/fake_news_detection')
DATA_DIR = DRIVE_PROJECT / 'data' / 'Combined_Corpus_Test_Klasörü'
print('Test verisi:', DATA_DIR)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Klasör bulunamadı: {DATA_DIR}. Drive yolunu kontrol edin.')
print('CSV dosyaları:', [p.name for p in sorted(DATA_DIR.glob('*.csv'))])


In [ ]:
import subprocess, sys
PROJECT = Path('/content/fake-news-detection')
if (PROJECT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
elif not PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/burak-dursunn/fake-news-detection.git', str(PROJECT)], check=True)
else:
    raise RuntimeError(f'Git deposu olmayan klasör: {PROJECT}')
print('Kod sürümü:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-colab.txt')], check=True)


## Eğitilmiş model

Paylaştığınız Reuters çıkarılmış ISOT modeli kullanılır. Test metinlerindeki Reuters ifadeleri korunur.


In [ ]:
MODEL_DIR = DRIVE_PROJECT / 'sonuclar' / 'ISOT' / 'ISOT_02_DistilBERT_256Token_Reuters_Silinen' / 'en_iyi_model'
if not (MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError(f'Eğitilmiş model bulunamadı: {MODEL_DIR}. Drive yolunu kontrol edin.')
print('Kullanılacak model:', MODEL_DIR)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab menüsünden Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU seçin.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from datetime import datetime
# Her çalıştırma ayrı klasör açar; önceki sonuçların üzerine yazılmaz.
OUTPUT = DRIVE_PROJECT / 'sonuclar' / 'Combined_Corpus_Test' / ('ISOT_DistilBERT_256Token_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
command = [sys.executable, '-u', '-m', 'src.evaluate_combined_corpus',
           '--data-dir', str(DATA_DIR), '--model', str(MODEL_DIR),
           '--output', str(OUTPUT), '--max-length', '256', '--batch-size', '16']
print('Test başlatılıyor. Sonuç klasörü:', OUTPUT, flush=True)
subprocess.run(command, cwd=PROJECT, check=True)


In [ ]:
from IPython.display import Image, display
for path in sorted(OUTPUT.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))
print('Tüm sonuçlar:', OUTPUT)
